# Main

In [41]:
# importing packages
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [119]:

# -------------------------------------------------
# Define absolute paths to the three source files
# -------------------------------------------------
ROOT = "/Users/marierobillon/LocalFiles/DS_causal_inf_for_sustainability_teamwork"
RAW = f"{ROOT}/data/data/raw"
ELCOM_CSV   = f"{RAW}/elcom_tariffs_H2_H4_H7.csv"
BFE_PLANTS  = f"{RAW}/bfe_plants.csv/ElectricityProductionPlant.csv"
DWELLINGS   = f"{RAW}/bfs_dwellings_by_occupant_2000_all.csv"
CROSSWALK  = f"{RAW}/bfs_commune_levels_2025.csv"

# Data

## Load and first cleaning

### X variable

In [ ]:

# Load **X** – ElCom household tariffs
# -------------------------------------------------
elcom_raw = pd.read_csv(ELCOM_CSV, sep=",", low_memory=False)

elcom_raw.head()


,period,municipality,operator,category,total,energy,grid,charge,aidfee,fixcosts
0,2018,3921,10,H2,25.359,7.765,13.194,2.10,2.3,108.0
1,2018,4045,804,H2,17.801,4.264,10.387,0.85,2.3,102.0
2,2018,5590,698,H2,25.480,8.900,12.350,1.93,2.3,60.0
3,2018,979,559,H2,22.802,6.763,12.989,0.75,2.3,126.0
4,2018,4080,594,H2,20.678,4.798,12.230,1.35,2.3,147.6


In [ ]:
# Keep only the columns we need for X
# -----------------------------------
cols_needed = ["municipality", "period", "operator", "category", "total", "energy"]
elcom = elcom_raw[cols_needed].copy()

# Rename to the semantic names we’ll use later
elcom = elcom.rename(columns={
    "municipality":   "commune_number",   # old commune ID
    "period":         "year",             # tariff year (int)
    "operator":       "grid_operator",    # operator code
    "category":       "tariff_profile",   # H2 / H4 / H7, H4 correspond to swiss households
    "total":          "total_ct_per_kwh", # total H4 tariff (cent/kWh), sum of energy, grid, charge and aidfree colums
    "energy":         "energy_ct_per_kwh" # energy component (cent/kWh)
})

elcom.head()

,commune_number,year,grid_operator,tariff_profile,total_ct_per_kwh,energy_ct_per_kwh
0,3921,2018,10,H2,25.359,7.765
1,4045,2018,804,H2,17.801,4.264
2,5590,2018,698,H2,25.480,8.900
3,979,2018,559,H2,22.802,6.763
4,4080,2018,594,H2,20.678,4.798


In [52]:
# Only keeping the tariff_profile H4 because it corresponds to swiss households

elcom_h4 = elcom[elcom["tariff_profile"] == "H4"].copy()

print(elcom.shape)
print(elcom_h4.shape)


(126086, 6)
(42044, 6)


### Y Variable

In [77]:
# -------------------------------------------------
# Load **Y** – SFOE plant register (solar plants only)
# -------------------------------------------------
plants = pd.read_csv(BFE_PLANTS, low_memory=False)

plants.head()


,xtf_id,Address,PostCode,Municipality,Canton,EGID,BeginningOfOperation,InitialPower,TotalPower,MainCategory,SubCategory,PlantCategory,_x,_y
0,5686,Binenweg 5,3904,Naters,VS,892091.0,1969-09-01,349576.0,349576.0,maincat_1,subcat_1,plantcat_7,2642480.025,1130235.125
1,5726,Robbia 2,7741,S. Carlo (Poschiavo),GR,9080323.0,2024-01-16,38250.0,38250.0,maincat_1,subcat_1,plantcat_4,NaN,NaN
2,5727,NaN,7744,Campocologno,GR,NaN,1907-03-01,55000.0,55000.0,maincat_1,subcat_1,plantcat_7,2808646.000,1123676.000
3,5730,Büdemji,7240,Küblis,GR,NaN,1922-01-01,44200.0,44200.0,maincat_1,subcat_1,plantcat_7,2778481.000,1198505.000
4,5740,Auenstrasse,8783,Linthal,GL,NaN,1962-03-18,1526000.0,1526000.0,maincat_1,subcat_1,plantcat_6,NaN,NaN


In [78]:
# Keep only the columns we need for Y
# -----------------------------------

renaming_map = {
    "xtf_id"               : "plant_id",          # unique identifier for the plant
    "Address"              : "address",           # street address of the plant
    "PostCode"             : "postcode",          # postal code
    "Municipality"         : "commune_number",    # old commune ID (matches tariff & census)
    "Canton"               : "canton",            # Canton abbreviation (e.g. VS, GR)
    "EGID"                 : "egid",              # official land‑registry identifier
    "BeginningOfOperation": "commission_date",# date the plant entered service
    "InitialPower"         : "initial_power",     # power at start (kW) – rarely used
    "TotalPower"           : "capacity_kw",       # current installed capacity (kW)
    "MainCategory"         : "main_category",     # high‑level plant class
    "SubCategory"          : "sub_category",      # detailed plant class (subcat_2 = solar)
    "PlantCategory"        : "plant_category",    # mounting type (attached, integrated, free‑standing)
    "_x"                   : "coord_x_lv95",      # Swiss LV95 easting
    "_y"                   : "coord_y_lv95"       # Swiss LV95 northing
}
plants = plants.rename(columns=renaming_map)


In [82]:
# Extract year from commissioning date (format: YYYY‑MM‑DD or YYYY)
plants["commission_year"] = pd.to_datetime(plants["commission_date"], errors="coerce").dt.year
# We need: commune_number (or postcode → commune), commissioning year,capacity (kW), and coordinates (_x, _y) for spatial checks.
cols_needed_y = ["commune_number", "commission_year", "capacity_kw", "coord_x_lv95", "coord_y_lv95","sub_category"]
plants = plants[cols_needed_y]

plants.head()

,commune_number,commission_year,capacity_kw,coord_x_lv95,coord_y_lv95,sub_category
0,Naters,1969,349576.0,2642480.025,1130235.125,subcat_1
1,S. Carlo (Poschiavo),2024,38250.0,NaN,NaN,subcat_1
2,Campocologno,1907,55000.0,2808646.000,1123676.000,subcat_1
3,Küblis,1922,44200.0,2778481.000,1198505.000,subcat_1
4,Linthal,1962,1526000.0,NaN,NaN,subcat_1


In [83]:
# Keep only solar plants (SubCategory code = 'subcat_2')
solar = plants[plants["sub_category"] == "subcat_2"].copy()

print(plants.shape)
print(solar.shape)


(339499, 6)
(337455, 6)


### Z variable

For each commune, the number of dwellings for a specific number of rooms and occupancy

In [112]:
# -------------------------------------------------
#  Load **Z** – Census 2000 dwellings by occupant
# -------------------------------------------------
dwell = pd.read_csv(DWELLINGS, low_memory=False)

dwell.head()


,Gemeinde,Anzahl Zimmer,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in
0,Schweiz,1 Zimmer,139953,4015,5579,3752,1366,757,916,190
1,Schweiz,2 Zimmer,327795,15544,18692,13618,5510,1149,4561,466
2,0001 Aeugst am Albis,1 Zimmer,15,0,1,1,1,1,0,0
3,0001 Aeugst am Albis,2 Zimmer,53,0,1,1,0,0,0,0
4,0002 Affoltern am Albis,1 Zimmer,81,6,3,5,0,0,0,0


In [113]:
# Romoving the lines with the sum of specific dwellings for whole switzerland
dwell = dwell[dwell["Gemeinde"] != "Schweiz"]

# Extract only the 4‑digit code from the combined string
dwell["commune_number"] = dwell["Gemeinde"].str.extract(r"(\d{4})").astype(int)
dwell = dwell.drop(columns=["Gemeinde"])

dwell.head()


,Anzahl Zimmer,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in,commune_number
2,1 Zimmer,15,0,1,1,1,1,0,0,1
3,2 Zimmer,53,0,1,1,0,0,0,0,1
4,1 Zimmer,81,6,3,5,0,0,0,0,2
5,2 Zimmer,396,36,10,8,0,2,2,0,2
6,1 Zimmer,16,0,2,3,1,0,0,0,3


In [114]:
# Compute the total dwellings per commune
# ---------------------------------------
# We keep the nine count columns (exclude the first and the last):
count_cols = [
    "Mieter/in",
    "Genossenschafter/in",
    "Stockwerk-/Wohnungseigentümer/in",
    "Alleineigentümer/in des Hauses",
    "Miteigentümer/in des Hauses",
    "Inhaber/in einer Dienstwohnung",
    "Inhaber/in einer Freiwohnung",
    "Pächter/in"
]

# sum all rows that belong to the same commune
# This collapses the “by‑room‑size” dimension.
dwell_agg = (
    dwell.groupby("commune_number", as_index=False)[count_cols]
          .sum()
)

# Row‑wise total dwellings for each commune (after aggregation)
dwell_agg["total_dwellings_2000"] = dwell_agg[count_cols].sum(axis=1)

dwell_agg.head()

,commune_number,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in,total_dwellings_2000
0,1,225,0,55,266,29,5,2,6,588
1,2,2596,247,225,804,143,25,12,4,4056
2,3,701,64,140,495,59,11,10,1,1481
3,4,494,2,85,475,64,3,10,10,1143
4,5,499,51,81,444,56,4,5,3,1143


In [115]:
#  Compute Owner‑occupied count and share (Z variable)
# ------------------------------------------------------------------
dwell_agg["owner_occupied_count"] = (
    dwell_agg["Alleineigentümer/in des Hauses"] + dwell_agg["Miteigentümer/in des Hauses"]
)
dwell_agg["owner_occupied_share"] = (
    dwell_agg["owner_occupied_count"] / dwell_agg["total_dwellings_2000"]
)

dwell_agg.head()

,commune_number,Mieter/in,Genossenschafter/in,Stockwerk-/Wohnungseigentümer/in,Alleineigentümer/in des Hauses,Miteigentümer/in des Hauses,Inhaber/in einer Dienstwohnung,Inhaber/in einer Freiwohnung,Pächter/in,total_dwellings_2000,owner_occupied_count,owner_occupied_share
0,1,225,0,55,266,29,5,2,6,588,295,0.501701
1,2,2596,247,225,804,143,25,12,4,4056,947,0.233481
2,3,701,64,140,495,59,11,10,1,1481,554,0.374072
3,4,494,2,85,475,64,3,10,10,1143,539,0.471566
4,5,499,51,81,444,56,4,5,3,1143,500,0.437445


In [117]:
# Only keep the columns of Z needed
# ---------------------------------

cols_needed_z = ["total_dwellings_2000", "owner_occupied_count", "owner_occupied_share", "commune_number"]
dwell_agg = dwell_agg[cols_needed_z]

dwell_agg.head()

,total_dwellings_2000,owner_occupied_count,owner_occupied_share,commune_number
0,588,295,0.501701,1
1,4056,947,0.233481,2
2,1481,554,0.374072,3
3,1143,539,0.471566,4
4,1143,500,0.437445,5


## Prepare to join datasets by updated commune numbers

In [131]:
# -------------------------------------------------
# Helper: map old commune numbers → 2026 numbers
# -------------------------------------------------

cross = pd.read_csv(CROSSWALK, low_memory=False) # contains the key for commune conversion from historical to new

cross.columns = (
    cross.columns.str.strip()
                 .str.lower()
                .str.replace(" ", "_")
                .str.replace(r"[^\w]", "", regex=True)
)

cross.head()


,historicalcode,bfscode,name,cantonid,canton,districtid,district,fua2014,degurb2021,fua2021,...,sprgeb2020,agksa2020,aggl2020,agglgk2020,mont2019,stalan2020,gdetyp2020_9,gdetyp2020_25,gbae2018,bae2018
0,10009,4122,Villnachern,19,Aargau,10023,Bezirk Brugg,0,2,0,...,1,0,0,0,0,3,23,236,12,12010
1,10078,6158,Vionnaz,23,Valais / Wallis,10013,District de Monthey,0,3,5586960,...,2,0,0,0,1,2,22,226,2,2040
2,10157,3023,Speicher,15,Appenzell Ausserrhoden,10098,Bezirk Mittelland,3203960,2,3203960,...,1,3203910,3203,3,1,2,21,217,15,15050
3,10159,6011,Zwischbergen,23,Valais / Wallis,10035,Bezirk Brig,0,3,0,...,1,0,0,0,1,3,33,338,7,7090
4,10162,2228,Villars-sur-Glâne,10,Fribourg / Freiburg,10104,District de la Sarine,0,1,2196960,...,2,2196910,2196,4,0,1,12,122,4,4030


In [141]:

elcom_min, elcom_max = elcom["commune_number"].min(), elcom["commune_number"].max()
dwell_min, dwell_max = dwell_agg["commune_number"].min(), dwell_agg["commune_number"].max()
cross_min, cross_max = cross["bfscode"].min(), cross["bfscode"].max()

# Build a one‑row‑per‑dataset pandas DataFrame
summary = pd.DataFrame({
    "Dataset": ["elcom", "solar", "dwell_agg","cross"],
    "Commune ID range": [
        f"{elcom_min} – {elcom_max}",
        "NA",
        f"{dwell_min} – {dwell_max}",
        f"{cross_min} – {cross_max}"
    ],
    "Number of distinct IDs": [
        elcom["commune_number"].nunique(),
        solar["commune_number"].nunique(),
        dwell_agg["commune_number"].nunique(),
        cross["bfscode"].nunique()
    ]
})

# Print (or display) the whole table in one output cell
print("Commune‑number overview")
display(summary)          # `display` shows a nice HTML table in Jupyter

Commune‑number overview


,Dataset,Commune ID range,Number of distinct IDs
0,elcom,1 – 9998,2715
1,solar,NA,3894
2,dwell_agg,1 – 6806,2896
3,cross,1 – 6812,2121


In [ ]:
# # Replace old IDs with the new ones
for df in [elcom, solar, dwell]:
    df["commune_id"] = df["new_commune_number_2026"].fillna(df["commune_number"])
    df.drop(columns=["commune_number", "old_commune_number", "new_commune_number_2026"], inplace=True)
# -------------------------------------------------
# 7️⃣  Compute **X** – Change in H4 total tariff (2023 – 2022)
# -------------------------------------------------
# First average tariff across operators for each commune‑year.
x_avg = (
    elcom.groupby(["commune_id", "year"])["h4_total"]
         .mean()
         .reset_index(name="h4_total_avg")
)
# Pivot to have 2022 and 2023 side‑by‑side
x_pivot = x_avg.pivot(index="commune_id", columns="year", values="h4_total_avg")
# Keep only 2022 & 2023 (if missing, result will be NaN)
x_pivot = x_pivot[[2022, 2023]] if {2022, 2023}.issubset(x_pivot.columns) else x_pivot
# X = 2023 – 2022
x_pivot["X"] = x_pivot[2023] - x_pivot[2022]
X = x_pivot["X"].rename("X")          # Series indexed by commune_id

In [ ]:

# -------------------------------------------------
# 8️⃣  Compute **Y** – New solar installations per commune‑year
# -------------------------------------------------
y_counts = (
    solar.groupby(["commune_id", "commission_year"])
         .size()
         .reset_index(name="new_solar_plants")
)
# If you also want total capacity per year:
y_capacity = (
    solar.groupby(["commune_id", "commission_year"])["capacity_kw"]
         .sum()
         .reset_index(name="solar_capacity_kw")
)
Y = y_counts.pivot(index="commune_id", columns="commission_year", values="new_solar_plants")
# Optional: keep capacity side‑by‑side as a second DataFrame
Y_capacity = y_capacity.pivot(index="commune_id", columns="commission_year", values="solar_capacity_kw")

# -------------------------------------------------
# 10️⃣  Assemble a tidy DataFrame with X, Y, Z side‑by‑side
# -------------------------------------------------
final = pd.concat([X, Z], axis=1)               # X and Z are Series
# For Y we keep the yearly counts; you can also collapse to a single metric.
# Example: total new solar plants from 2020‑2026
if Y is not None:
    y_total = Y.loc[:, 2020:2026].sum(axis=1).rename("Y_total_new_solar_2020_2026")
    final = final.join(y_total, how="left")
print("✅ Finished loading X, Y, Z")
display(final.head())

## Data visualisation